# Quiet Relay: 2026 - Updated Game Runner

This notebook launches the current data-driven game files from Jupyter:

- `quiet_relay_terminal_datadriven.py`
- `quiet_relay_vertical_slice_datadriven.py`

## What this runner matches
- per-turn Power / Precision / Composure input in manual battle flow
- AP-based player turns and battle loadouts
- starting equipment, equipment bonuses, and healing potions
- updated expedition / vertical-slice flow

## How to use
1. Run the setup cell.
2. Edit the configuration cell.
3. Run the game cell.
4. Use the artifact cell to inspect the latest log, save, or run report.

The default configuration opens the data-driven vertical slice in solo mode.

## Manual mode notes
- `terminal_datadriven` manual play will ask for a 3-action extra loadout at battle start.
- It will then ask for Power / Precision / Composure once at the start of each player turn.
- During that turn, it will keep asking for actions while AP remains.
- `vertical_slice_datadriven` manual play can also ask for node calibration and route choices.

`AUTO_MODE = False` is now the default so the notebook opens in manual play.
Set `AUTO_MODE = True` when you want a smooth autoplay smoke run.


In [1]:
from pathlib import Path
import importlib.util
import json
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

GAME_CONFIGS = {
    'terminal_datadriven': {
        'label': 'Updated terminal combat core',
        'script': 'quiet_relay_terminal_datadriven.py',
        'kind': 'battle',
        'default_log_file': 'quiet_relay_last_battle_log_datadriven.txt',
    },
    'vertical_slice_datadriven': {
        'label': 'Updated expedition vertical slice',
        'script': 'quiet_relay_vertical_slice_datadriven.py',
        'kind': 'campaign',
        'default_save_file': 'quiet_relay_vertical_slice_datadriven_save.json',
        'default_log_dir': 'quiet_relay_vertical_slice_datadriven_logs',
        'default_run_report': 'quiet_relay_vertical_slice_datadriven_last_run.txt',
    },
}


def to_project_path(value, default_name=None):
    if value in (None, ''):
        if default_name is None:
            return None
        return PROJECT_ROOT / default_name
    path = Path(value)
    return path if path.is_absolute() else PROJECT_ROOT / path


def resolve_run_paths(game_key, log_file=None, save_file=None, log_dir=None):
    config = GAME_CONFIGS[game_key]
    paths = {'script': PROJECT_ROOT / config['script']}
    if config['kind'] == 'battle':
        paths['log_file'] = to_project_path(log_file, config['default_log_file'])
    else:
        paths['save_file'] = to_project_path(save_file, config['default_save_file'])
        paths['log_dir'] = to_project_path(log_dir, config['default_log_dir'])
        paths['run_report'] = paths['save_file'].with_name(config['default_run_report'])
    return paths


def ensure_output_dirs(paths):
    for key in ('log_file', 'save_file', 'run_report'):
        path = paths.get(key)
        if path is not None:
            path.parent.mkdir(parents=True, exist_ok=True)
    log_dir = paths.get('log_dir')
    if log_dir is not None:
        log_dir.mkdir(parents=True, exist_ok=True)


def load_game_module(game_key, paths):
    script_path = paths['script']
    if not script_path.exists():
        raise FileNotFoundError(f'Could not find: {script_path}')
    module_name = f"{script_path.stem}_nb"
    if module_name in sys.modules:
        del sys.modules[module_name]
    spec = importlib.util.spec_from_file_location(module_name, script_path)
    if spec is None or spec.loader is None:
        raise ImportError(f'Could not load module spec for: {script_path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


def prepare_module_defaults(module, paths):
    for attr, key in (
        ('DEFAULT_SAVE_FILE', 'save_file'),
        ('DEFAULT_LOG_DIR', 'log_dir'),
        ('DEFAULT_RUN_REPORT', 'run_report'),
    ):
        if hasattr(module, attr) and key in paths:
            setattr(module, attr, str(paths[key]))


def write_axis_file(path, default=(60, 60, 60), nodes=None):
    path = to_project_path(path)
    nodes = nodes or {}
    payload = {
        'default': {
            'power': default[0],
            'precision': default[1],
            'composure': default[2],
        },
        'nodes': nodes,
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    print(f'Wrote axis file: {path}')
    return path


def build_argv(
    game_key,
    paths,
    *,
    auto_mode=False,
    seed=2026,
    party='vanguard',
    scenario='bell_warden',
    load_existing=False,
    fresh_start=False,
    district=None,
    solo_character='vanguard',
    node_axis=None,
    axis_file=None,
    auto_route='first',
):
    if game_key == 'terminal_datadriven':
        argv = [
            '--party', str(party),
            '--scenario', str(scenario),
            '--seed', str(seed),
            '--log-file', str(paths['log_file']),
        ]
        if auto_mode:
            argv.append('--auto')
        return argv

    argv = [
        '--save-file', str(paths['save_file']),
        '--log-dir', str(paths['log_dir']),
        '--seed', str(seed),
    ]
    if auto_mode:
        argv.append('--auto')
    if load_existing:
        argv.append('--load')
    if fresh_start:
        argv.append('--fresh')
    if district not in (None, ''):
        argv.extend(['--district', str(district)])
    if solo_character not in (None, ''):
        argv.extend(['--solo-character', str(solo_character)])
    if axis_file not in (None, ''):
        argv.extend(['--axis-file', str(to_project_path(axis_file))])
    elif node_axis is not None:
        power, precision, composure = node_axis
        argv.extend(['--node-axis', str(power), str(precision), str(composure)])
    if auto_route not in (None, ''):
        argv.extend(['--auto-route', str(auto_route)])
    return argv


def print_available_games():
    print('Available GAME_KEY values:')
    for key, config in GAME_CONFIGS.items():
        print(f" - {key:<26} -> {config['label']} ({config['script']})")


def describe_interactive_flow(game_key, auto_mode):
    if auto_mode:
        print('Mode note: AUTO mode avoids prompts and is the smoothest notebook flow.')
        return
    if game_key == 'terminal_datadriven':
        print('Manual note: expect a 3-action loadout prompt at battle start,')
        print('then Power / Precision / Composure once per player turn, then AP action prompts.')
        return
    print('Manual note: expedition play can ask for node calibration, routes, battles, and rewards.')


def print_configuration(
    game_key,
    paths,
    *,
    auto_mode,
    seed,
    party,
    scenario,
    load_existing,
    fresh_start,
    district,
    solo_character,
    node_axis,
    axis_file,
    auto_route,
):
    config = GAME_CONFIGS[game_key]
    print(f"GAME_KEY    = {game_key} ({config['label']})")
    print(f"SCRIPT      = {paths['script']}")
    print(f"AUTO_MODE   = {auto_mode}")
    print(f"SEED        = {seed}")
    if config['kind'] == 'battle':
        print(f"PARTY       = {party}")
        print(f"SCENARIO    = {scenario}")
        print(f"LOG_FILE    = {paths['log_file']}")
        describe_interactive_flow(game_key, auto_mode)
        return
    print(f"LOAD_EXIST  = {load_existing}")
    print(f"FRESH_START = {fresh_start}")
    print(f"SAVE_FILE   = {paths['save_file']}")
    print(f"LOG_DIR     = {paths['log_dir']}")
    print(f"RUN_REPORT  = {paths['run_report']}")
    print(f"DISTRICT    = {district}")
    print(f"SOLO_CHAR   = {solo_character}")
    print(f"NODE_AXIS   = {node_axis}")
    print(f"AXIS_FILE   = {to_project_path(axis_file) if axis_file not in (None, '') else None}")
    print(f"AUTO_ROUTE  = {auto_route}")
    describe_interactive_flow(game_key, auto_mode)


def run_game(
    game_key,
    *,
    auto_mode=False,
    seed=2026,
    party='vanguard',
    scenario='bell_warden',
    log_file=None,
    load_existing=False,
    fresh_start=False,
    save_file=None,
    log_dir=None,
    district=None,
    solo_character='vanguard',
    node_axis=None,
    axis_file=None,
    auto_route='first',
):
    paths = resolve_run_paths(game_key, log_file=log_file, save_file=save_file, log_dir=log_dir)
    ensure_output_dirs(paths)
    module = load_game_module(game_key, paths)
    prepare_module_defaults(module, paths)
    argv = build_argv(
        game_key,
        paths,
        auto_mode=auto_mode,
        seed=seed,
        party=party,
        scenario=scenario,
        load_existing=load_existing,
        fresh_start=fresh_start,
        district=district,
        solo_character=solo_character,
        node_axis=node_axis,
        axis_file=axis_file,
        auto_route=auto_route,
    )
    print('Running:')
    print(f"  {GAME_CONFIGS[game_key]['label']}")
    print(f"  script = {paths['script']}")
    print(f"  argv   = {argv}")
    print('-' * 80)
    rc = module.main(argv)
    print('-' * 80)
    print(f'Exit code: {rc}')
    return {
        'game_key': game_key,
        'paths': paths,
        'argv': argv,
        'exit_code': rc,
    }


def show_text_file(path):
    path = Path(path)
    if not path.exists():
        print(f'No file found at: {path}')
        return
    print(path.read_text(encoding='utf-8'))


def show_json_file(path):
    path = Path(path)
    if not path.exists():
        print(f'No file found at: {path}')
        return
    payload = json.loads(path.read_text(encoding='utf-8'))
    print(json.dumps(payload, indent=2))


def show_latest_log(log_dir):
    log_dir = Path(log_dir)
    if not log_dir.exists():
        print(f'No log directory found at: {log_dir}')
        return
    logs = sorted(log_dir.glob('*.txt'))
    if not logs:
        print(f'No log files in: {log_dir}')
        return
    latest = logs[-1]
    print(f'Latest log: {latest}')
    print('-' * 80)
    print(latest.read_text(encoding='utf-8'))


def show_default_artifacts(last_run):
    if not last_run:
        print('No LAST_RUN record yet. Run the game cell first.')
        return
    paths = last_run['paths']
    for key in ('log_file', 'save_file', 'log_dir', 'run_report'):
        if key in paths:
            print(f"{key:<10} = {paths[key]}")
    print('-' * 80)
    if 'log_file' in paths:
        show_text_file(paths['log_file'])
        return
    show_text_file(paths['run_report'])


print(f'Project root: {PROJECT_ROOT}')
print_available_games()


Project root: C:\Users\kim50\projects\trbg_desktop
Available GAME_KEY values:
 - terminal_datadriven        -> Updated terminal combat core (quiet_relay_terminal_datadriven.py)
 - vertical_slice_datadriven  -> Updated expedition vertical slice (quiet_relay_vertical_slice_datadriven.py)


In [2]:
# Edit these values, then run the game cell below.

GAME_KEY = 'vertical_slice_datadriven'
AUTO_MODE = False
SEED = 2026

# Battle runner options
PARTY = 'vanguard'
SCENARIO = 'bell_warden'
LOG_FILE = None

# Expedition runner options
LOAD_EXISTING = False
FRESH_START = True
SAVE_FILE = None
LOG_DIR = None
DISTRICT = None
SOLO_CHARACTER = 'vanguard'
NODE_AXIS = None
AXIS_FILE = None
AUTO_ROUTE = 'first'

CURRENT_PATHS = resolve_run_paths(
    GAME_KEY,
    log_file=LOG_FILE,
    save_file=SAVE_FILE,
    log_dir=LOG_DIR,
)

print_configuration(
    GAME_KEY,
    CURRENT_PATHS,
    auto_mode=AUTO_MODE,
    seed=SEED,
    party=PARTY,
    scenario=SCENARIO,
    load_existing=LOAD_EXISTING,
    fresh_start=FRESH_START,
    district=DISTRICT,
    solo_character=SOLO_CHARACTER,
    node_axis=NODE_AXIS,
    axis_file=AXIS_FILE,
    auto_route=AUTO_ROUTE,
)


GAME_KEY    = vertical_slice_datadriven (Updated expedition vertical slice)
SCRIPT      = C:\Users\kim50\projects\trbg_desktop\quiet_relay_vertical_slice_datadriven.py
AUTO_MODE   = False
SEED        = 2026
LOAD_EXIST  = False
FRESH_START = True
SAVE_FILE   = C:\Users\kim50\projects\trbg_desktop\quiet_relay_vertical_slice_datadriven_save.json
LOG_DIR     = C:\Users\kim50\projects\trbg_desktop\quiet_relay_vertical_slice_datadriven_logs
RUN_REPORT  = C:\Users\kim50\projects\trbg_desktop\quiet_relay_vertical_slice_datadriven_last_run.txt
DISTRICT    = None
SOLO_CHAR   = vanguard
NODE_AXIS   = (80, 80, 80)
AXIS_FILE   = None
AUTO_ROUTE  = first
Manual note: expedition play can ask for node calibration, routes, battles, and rewards.


In [ ]:
LAST_RUN = run_game(
    GAME_KEY,
    auto_mode=AUTO_MODE,
    seed=SEED,
    party=PARTY,
    scenario=SCENARIO,
    log_file=LOG_FILE,
    load_existing=LOAD_EXISTING,
    fresh_start=FRESH_START,
    save_file=SAVE_FILE,
    log_dir=LOG_DIR,
    district=DISTRICT,
    solo_character=SOLO_CHARACTER,
    node_axis=NODE_AXIS,
    axis_file=AXIS_FILE,
    auto_route=AUTO_ROUTE,
)

LAST_RUN


Running:
  Updated expedition vertical slice
  script = C:\Users\kim50\projects\trbg_desktop\quiet_relay_vertical_slice_datadriven.py
  argv   = ['--save-file', 'C:\\Users\\kim50\\projects\\trbg_desktop\\quiet_relay_vertical_slice_datadriven_save.json', '--log-dir', 'C:\\Users\\kim50\\projects\\trbg_desktop\\quiet_relay_vertical_slice_datadriven_logs', '--seed', '2026', '--fresh', '--solo-character', 'vanguard', '--node-axis', '80', '80', '80', '--auto-route', 'first']
--------------------------------------------------------------------------------

🕯️ Quiet Relay: 2026 - Operations Base [Fully Data-Driven]
------------------------------------------------------------------------------------
Late 2026. Rain ticks against emergency floodlights while the expedition team
regroups in a ruined signal base. Beyond the shutters waits the Rain Toll
Corridor, a short but hostile district warped by bells, transit steel, and ash.

🧭 District: District 03: Rain Toll Corridor
Last result: No expedit

In [ ]:
# Show the default artifact for the last run.
# Battle mode: last battle log.
# Expedition mode: last run report.
show_default_artifacts(LAST_RUN)


## Quick presets

Battle auto smoke:

```python
GAME_KEY = 'terminal_datadriven'
AUTO_MODE = True
PARTY = 'vanguard'
SCENARIO = 'skirmish'
```

Battle manual test of the AP / loadout flow:

```python
GAME_KEY = 'terminal_datadriven'
AUTO_MODE = False
PARTY = 'vanguard,duelist,cantor'
SCENARIO = 'quiet_magistrate'
```

Expedition auto smoke with fixed node axes:

```python
GAME_KEY = 'vertical_slice_datadriven'
AUTO_MODE = False
LOAD_EXISTING = False
FRESH_START = True
SOLO_CHARACTER = 'vanguard'
NODE_AXIS = None
AUTO_ROUTE = 'first'
```

Expedition manual calibration run:

```python
GAME_KEY = 'vertical_slice_datadriven'
AUTO_MODE = False
LOAD_EXISTING = False
FRESH_START = True
SOLO_CHARACTER = 'vanguard'
NODE_AXIS = None
AXIS_FILE = None
```
